<a href="https://colab.research.google.com/github/khushii-vermaa/UCS420/blob/main/Assignment4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



**ASSIGNMNET** 4

In [3]:
# QUESTION 1
import pandas as pd

rollno = input("Enter roll no.: ")

last_digs = rollno[-2:]

categ = ["billing", "account", "general"]

fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "categ": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "categ": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "categ": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "categ": "billing"
    }
]

personalized_entries = []

for dig in last_digs:
    d = int(dig)
    category = categ[d % 3]

    if category == "billing":
        question = "how can i check my payment status"
        answer = "You can check your payment status from the Billing section."
        keywords = "payment status billing transaction"

    elif category == "account":
        question = "how do i update my registered mobile number"
        answer = "Go to Account Settings and update your registered mobile number."
        keywords = "mobile number update account"

    else:
        question = "how can i contact customer support"
        answer = "You can contact customer support during working hours."
        keywords = "support help contact"

    personalized_entries.append({
        "question": question,
        "answer": answer,
        "keywords": keywords,
        "category": category
    })

all_entries = fixed_entries + personalized_entries

df = pd.DataFrame(all_entries)
print(df)



Enter roll no.: 1024170065
                             question  \
0              what is the annual fee   
1               how to reset password   
2         what are your working hours   
3               how can i pay the fee   
4   how can i check my payment status   
5  how can i contact customer support   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4  You can check your payment status from the Bil...   
5  You can contact customer support during workin...   

                             keywords    categ category  
0               fee cost price charge  billing      NaN  
1                password reset login  account      NaN  
2              hours timing open time  general      NaN  
3                 pay payment upi fee  billing      NaN  
4 

In [4]:
# QUESTION 2
def scoreQuery(query, df):
  queryWords = query.lower().split()
  results = []
  for idx, row in df.iterrows():
    keywords = row["keywords"].lower().split()
    score = sum(1 for word in queryWords if word in keywords)
    if score > 0:
      results.append({
                "index": idx,
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "score": score
            })
    results.sort(key=lambda x: x["score"], reverse=True)

    return results


print("\nQUERY SCORING")

query = input("\nEnter query: ")

results = scoreQuery(query, df)

if len(results) == 0:
    print("No matching FAQ found.")

else:
    print("\nMatching FAQs:")

    for result in results:
        print(
            "Question:", result["question"],
            "| Category:", result["category"],
            "| Score:", result["score"]
        )





QUERY SCORING

Enter query: fee

Matching FAQs:
Question: what is the annual fee | Category: nan | Score: 1


In [6]:
# QUESTION 3
def same_category(category_name, df):

    result = df[
        df["category"].str.lower() == category_name.lower()
    ]

    return result[["question", "category"]]


personalized_category = personalized_entries[0]["category"]

print("Selected category:", personalized_category)

print(same_category(personalized_category, df))

print()

Selected category: billing
                            question category
4  how can i check my payment status  billing



In [7]:
# QUESTION 4
entry_index = 0

print("Selected entry:")
print(df.loc[entry_index])
new_keyword = input("Enter a new keyword: ")

if new_keyword.strip() != "":

    old_keywords = df.loc[entry_index, "keywords"]

    df.loc[entry_index, "keywords"] = (
        old_keywords + " " + new_keyword
    )

print("\nUpdated entry:")
print(df.loc[entry_index])

filename = rollno + "_faq_data.csv"

df.to_csv(filename, index=False)

print("\nFile saved as:", filename)

print()

Selected entry:
question       what is the annual fee
answer      The annual fee is Rs 500.
keywords        fee cost price charge
categ                         billing
category                          NaN
Name: 0, dtype: object
Enter a new keyword: payment

Updated entry:
question           what is the annual fee
answer          The annual fee is Rs 500.
keywords    fee cost price charge payment
categ                             billing
category                              NaN
Name: 0, dtype: object

File saved as: 1024170065_faq_data.csv



In [8]:
# QUESTION 5
category_count = df.groupby("category").size()
print(category_count)

category
billing    1
general    1
dtype: int64


In [14]:
import re

def tokenize(text):
    text = text.lower()
    words = re.findall(r'\b\w+\b', text)
    return set(words)

def score_query_tie_safe(query, df):

    query_words = tokenize(query)

    results = []

    for index, row in df.iterrows():

        text = (
            str(row["question"]) + " " +
            str(row["answer"]) + " " +
            str(row["keywords"])
        )

        entry_words = tokenize(text)

        matched_words = query_words.intersection(entry_words)

        if len(query_words) > 0:
            confidence = len(matched_words) / len(query_words)
        else:
            confidence = 0

        if confidence > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "confidence": confidence
            })

    if len(results) == 0:
        return []

    max_score = max(
        result["confidence"]
        for result in results
    )

    best_matches = []

    for result in results:
        if result["confidence"] == max_score:
            best_matches.append(result)

    return best_matches

In [13]:
# query that produces a tie
query = "fee"

results = score_query_tie_safe(query, df)

print("Query:", query)

for result in results:
    print(
        "Confidence:", round(result["confidence"], 2),
        "| Category:", result["category"],
        "| Question:", result["question"]
    )

Query: fee
Confidence: 1.0 | Category: nan | Question: what is the annual fee
Confidence: 1.0 | Category: nan | Question: how can i pay the fee


In [15]:
# query that doesnt produces a tie
query = "password reset"

results = score_query_tie_safe(query, df)

print("Query:", query)

for result in results:
    print(
        "Confidence:", round(result["confidence"], 2),
        "| Category:", result["category"],
        "| Question:", result["question"]
    )

Query: password reset
Confidence: 1.0 | Category: nan | Question: how to reset password
